In [1]:
import pandas as pd
import numpy as np

# streetcar data
df1 = pd.read_csv('../raw/streetcar/streetcar_2020.csv')
df2 = pd.read_csv('../raw/streetcar/streetcar_2021.csv')
df3 = pd.read_csv('../raw/streetcar/streetcar_2022.csv')
df4 = pd.read_csv('../raw/streetcar/streetcar_2023.csv')
df5 = pd.read_csv('../raw/streetcar/streetcar_2024.csv')
df6 = pd.read_csv('../raw/streetcar/streetcar_2025.csv')

In [2]:
print(df1.info())
print(df2.info())
print(df3.info())
print(df4.info())
print(df5.info())
print(df6.info())

<class 'pandas.DataFrame'>
RangeIndex: 802 entries, 0 to 801
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Report Date  802 non-null    str    
 1   Route        802 non-null    int64  
 2   Time         802 non-null    str    
 3   Day          802 non-null    str    
 4   Location     801 non-null    str    
 5   Incident     802 non-null    str    
 6   Delay        802 non-null    int64  
 7   Gap          802 non-null    int64  
 8   Direction    802 non-null    str    
 9   Vehicle      791 non-null    float64
dtypes: float64(1), int64(3), str(6)
memory usage: 108.0 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 1018 entries, 0 to 1017
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Date       1018 non-null   str  
 1   Line       1016 non-null   str  
 2   Time       1018 non-null   str  
 3   Day        1018 non-null   str  
 4   Location

In [3]:
# 2020 has different column names and HH:MM:SS time format — rename to match the rest
df1 = df1.rename(columns={
    'Report Date': 'Date',
    'Route':       'Line',
    'Delay':       'Min Delay',
    'Gap':         'Min Gap',
    'Direction':   'Bound',
})

In [4]:
# 2025 has _id, Station, Code, and route name appended to Line number
df6 = df6.drop(columns=['_id'])

codes = pd.read_csv('../raw/streetcar/streetcar_code.csv')
code_map = dict(zip(codes['CODE'], codes['DESCRIPTION']))
df6['Incident'] = df6['Code'].map(code_map)
df6 = df6.drop(columns=['Code'])

df6 = df6.rename(columns={'Station': 'Location'})

# strip route name suffix e.g. "504 KING" -> 504
df6['Line'] = df6['Line'].str.extract(r'^(\d+)').astype('float')

In [5]:
dfs = [df1, df2, df3, df4, df5, df6]
if all(set(dfs[0].columns) == set(df.columns) for df in dfs):
    print("All DataFrames have the same columns.")
else:
    print("Some DataFrames have different columns.")
    for i, d in enumerate(dfs, 1):
        print(f"df{i}: {d.columns.tolist()}")

All DataFrames have the same columns.


In [6]:
df = pd.concat([df1, df2, df3, df4, df5, df6], ignore_index=True)

In [7]:
df

,Date,Line,Time,Day,Location,Incident,Min Delay,Min Gap,Bound,Vehicle
0,2020-01-01,504,01:23:00,Wednesday,King and Dufferin,Mechanical,8,16,E/B,4446.0
1,2020-01-01,512,04:40:00,Wednesday,Queen/Parliament,Mechanical,23,46,W/B,4541.0
2,2020-01-01,504,06:17:00,Wednesday,Roncesvalles and Queen,Mechanical,6,14,W/B,4576.0
3,2020-01-01,504,07:50:00,Wednesday,King/Church,Mechanical,10,20,W/B,4486.0
4,2020-01-01,504,08:09:00,Wednesday,Queeen/ Roncesvalles,Mechanical,8,16,E/B,4504.0
...,...,...,...,...,...,...,...,...,...,...
63874,2026-01-31,505.0,00:47,Saturday,KINGSTON LOOP,DISORDERLY PATRON,0,0,NaN,4492.0
63875,2026-01-31,506.0,01:30,Saturday,MAIN AND DANFORTH,UNSANITARY VEHICLE,7,15,E,4482.0
63876,2026-01-31,504.0,01:44,Saturday,SUNNYSIDE LOOP,INJURIED/ILL CUSTOMER â TRANSPORTED,10,20,W,4660.0
63877,2026-01-31,506.0,01:47,Saturday,COLLEGE AND CLINTON,AUTO FOUL RAIL,41,61,E,4495.0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63879 entries, 0 to 63878
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       63879 non-null  str    
 1   Line       63277 non-null  object 
 2   Time       63879 non-null  str    
 3   Day        63879 non-null  str    
 4   Location   63877 non-null  str    
 5   Incident   62294 non-null  str    
 6   Min Delay  63879 non-null  int64  
 7   Min Gap    63879 non-null  int64  
 8   Bound      53147 non-null  str    
 9   Vehicle    63868 non-null  float64
dtypes: float64(1), int64(2), object(1), str(6)
memory usage: 8.2+ MB


In [9]:
# Line is a core feature — rows with no route cannot be used for prediction
df[df['Line'].isnull()]

,Date,Line,Time,Day,Location,Incident,Min Delay,Min Gap,Bound,Vehicle
1021,2021-01-06,NaN,07:38,Wednesday,RONCESVALLES CARHOUSE,Overhead,0,0,NaN,4447.0
1314,2021-01-16,NaN,06:15,Saturday,RUSSELL YARD,Emergency Services,0,0,NaN,4408.0
2559,2022-01-09,NaN,02:30,Sunday,QUEEN AND SHERBOURNE,Cleaning - Unsanitary,15,30,W,8447.0
2837,2022-01-12,NaN,01:35,Wednesday,QUEEN AND SOHO,Utilized Off Route,10,20,W,8745.0
4123,2022-02-03,NaN,08:55,Thursday,LESLIE BARNS,General Delay,0,0,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...
63716,2026-01-29,NaN,00:24,Thursday,DUNCAN SUBSTATION,OTHER - PLANT,0,0,NaN,0.0
63762,2026-01-30,NaN,15:43,Friday,RUSSELL CARHOUSE,OVERHEAD DEFECT,0,0,NaN,0.0
63787,2026-01-30,NaN,23:44,Friday,CANNAUGHT AND QUEEN,AUTO FOUL RAIL,0,0,N,4431.0
63793,2026-01-30,NaN,01:29,Friday,BAY AND GERRARD,OTHER,0,0,N,4497.0


In [10]:
df = df.dropna(subset=['Line'])

In [11]:
df[df['Location'].isnull()]

,Date,Line,Time,Day,Location,Incident,Min Delay,Min Gap,Bound,Vehicle
90,2020-01-06,505,08:33:00,Monday,NaN,General Delay,4,7,B/W,NaN
34957,2024-01-25,507,07:51,Thursday,NaN,Mechanical,10,20,NaN,4591.0


In [12]:
# negligible fraction — drop with no meaningful loss
df = df.dropna(subset=['Location'])

In [13]:
print(f"Incident nulls: {df['Incident'].isnull().sum()} ({df['Incident'].isnull().mean():.1%})")

Incident nulls: 1566 (2.5%)


In [14]:
# Incident nulls come only from 2025 unmapped codes — drop since it's a key feature
df = df.dropna(subset=['Incident'])

In [15]:
df[['Min Delay', 'Min Gap', 'Vehicle']].isnull().sum()

Min Delay     0
Min Gap       0
Vehicle      10
dtype: int64

In [16]:
df = df.dropna(subset=['Vehicle', 'Min Delay', 'Min Gap'])
df.info()

<class 'pandas.DataFrame'>
Index: 61699 entries, 0 to 63878
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       61699 non-null  str    
 1   Line       61699 non-null  object 
 2   Time       61699 non-null  str    
 3   Day        61699 non-null  str    
 4   Location   61699 non-null  str    
 5   Incident   61699 non-null  str    
 6   Min Delay  61699 non-null  int64  
 7   Min Gap    61699 non-null  int64  
 8   Bound      51676 non-null  str    
 9   Vehicle    61699 non-null  float64
dtypes: float64(1), int64(2), object(1), str(6)
memory usage: 8.5+ MB


In [17]:
df['Bound'].value_counts(dropna=False)

Bound
W      19227
E      19147
NaN    10023
S       6290
N       6077
E/B      312
W/B      300
B        135
N/B       75
S/B       68
B/W       32
r          2
T          2
8          2
ew         1
bw         1
Q          1
C          1
5          1
`          1
1          1
Name: count, dtype: int64

In [18]:
# Bound is null for garage incidents and emergencies — fill rather than drop
df['Bound'] = df['Bound'].fillna('UNKNOWN')
df['Direction_known'] = (df['Bound'] != 'UNKNOWN').astype(int)

In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 61699 entries, 0 to 63878
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Date             61699 non-null  str    
 1   Line             61699 non-null  object 
 2   Time             61699 non-null  str    
 3   Day              61699 non-null  str    
 4   Location         61699 non-null  str    
 5   Incident         61699 non-null  str    
 6   Min Delay        61699 non-null  int64  
 7   Min Gap          61699 non-null  int64  
 8   Bound            61699 non-null  str    
 9   Vehicle          61699 non-null  float64
 10  Direction_known  61699 non-null  int64  
dtypes: float64(1), int64(3), object(1), str(6)
memory usage: 9.0+ MB


In [20]:
# 2020 stores time as HH:MM:SS; all other years use HH:MM
def normalize_time(t):
    t = str(t).strip()
    if len(t) == 8 and t[2] == ':' and t[5] == ':':   # HH:MM:SS
        return t[:5]
    return t   # already HH:MM

df['Time'] = df['Time'].apply(normalize_time)
df['Hour']   = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
df['Minute'] = pd.to_datetime(df['Time'], format='%H:%M').dt.minute
df = df.drop(columns=['Time'])

In [21]:
df['Date']      = pd.to_datetime(df['Date'], format='mixed')
df['Month']     = df['Date'].dt.month
df['Year']      = df['Date'].dt.year
df['DayOfWeek'] = df['Date'].dt.dayofweek
df = df.drop(columns=['Date'])

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 61699 entries, 0 to 63878
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Line             61699 non-null  object 
 1   Day              61699 non-null  str    
 2   Location         61699 non-null  str    
 3   Incident         61699 non-null  str    
 4   Min Delay        61699 non-null  int64  
 5   Min Gap          61699 non-null  int64  
 6   Bound            61699 non-null  str    
 7   Vehicle          61699 non-null  float64
 8   Direction_known  61699 non-null  int64  
 9   Hour             61699 non-null  int32  
 10  Minute           61699 non-null  int32  
 11  Month            61699 non-null  int32  
 12  Year             61699 non-null  int32  
 13  DayOfWeek        61699 non-null  int32  
dtypes: float64(1), int32(5), int64(3), object(1), str(4)
memory usage: 8.3+ MB


In [23]:
df['Bound'].unique()

<ArrowStringArray>
[    'E/B',     'W/B',     'S/B',     'N/B',     'B/W',      'ew',      'bw',
       'r',       'W', 'UNKNOWN',       'S',       'N',       'E',       'B',
       'T',       'Q',       'C',       '5',       '`',       '1',       '8']
Length: 21, dtype: str

In [24]:
DIRECTION_MAP = {
    # Northbound
    'N': 'NB', 'n': 'NB', 'NB': 'NB', 'nb': 'NB', 'N/B': 'NB',
    # Southbound
    'S': 'SB', 's': 'SB', 'SB': 'SB', 'S/B': 'SB',
    # Eastbound
    'E': 'EB', 'EB': 'EB', 'E/B': 'EB', 'e/b': 'EB', 'ew': 'EB',
    # Westbound
    'W': 'WB', 'WB': 'WB', 'wb': 'WB', 'W/B': 'WB',
    # Both / Bidirectional
    'B': 'BOTH', 'B/W': 'BOTH', 'bw': 'BOTH', 'btw': 'BOTH',
}

NOISE_VALUES = {'UNKNOWN', 'r', 'T', 'Q', 'C', '5', '`', '1', '8'}

def clean_direction(series: pd.Series) -> pd.Series:
    return (
        series
        .str.strip()
        .map(lambda x: DIRECTION_MAP.get(x, None) if x not in NOISE_VALUES else None)
    )

df['Direction_clean'] = clean_direction(df['Bound'])

unmapped = df[df['Direction_clean'].isna()]['Bound'].value_counts()
print("Unmapped values:\n", unmapped)

Unmapped values:
 Bound
UNKNOWN    10023
r              2
T              2
8              2
Q              1
C              1
5              1
`              1
1              1
Name: count, dtype: int64


In [25]:
# data integrity check
print(f"Rows retained: {len(df):,}")
df.head()

Rows retained: 61,699


,Line,Day,Location,Incident,Min Delay,Min Gap,Bound,Vehicle,Direction_known,Hour,Minute,Month,Year,DayOfWeek,Direction_clean
0,504,Wednesday,King and Dufferin,Mechanical,8,16,E/B,4446.0,1,1,23,1,2020,2,EB
1,512,Wednesday,Queen/Parliament,Mechanical,23,46,W/B,4541.0,1,4,40,1,2020,2,WB
2,504,Wednesday,Roncesvalles and Queen,Mechanical,6,14,W/B,4576.0,1,6,17,1,2020,2,WB
3,504,Wednesday,King/Church,Mechanical,10,20,W/B,4486.0,1,7,50,1,2020,2,WB
4,504,Wednesday,Queeen/ Roncesvalles,Mechanical,8,16,E/B,4504.0,1,8,9,1,2020,2,EB


In [26]:
df.to_csv('../clean_streetcar.csv', index=False)